# Point Cloud Classification with PointNet

 
**Dataset:** ModelNet10 — 10 classes of 3D CAD models (bathtub, bed, chair, desk, dresser, monitor, night_stand, sofa, table, toilet)


## 1. Environment Check

We confirm the Python version, PyTorch version, and the available hardware accelerator (CUDA for NVIDIA GPUs, MPS for Apple Silicon, or CPU fallback). The `device` variable detected here will be used throughout the project to move tensors onto the GPU.

In [1]:
import sys
import platform
import torch

print("Python version :", sys.version.split()[0])
print("Platform       :", platform.platform())
print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version   :", torch.version.cuda)
    print("GPU device     :", torch.cuda.get_device_name(0))
    print("GPU count      :", torch.cuda.device_count())
elif torch.backends.mps.is_available():
    print("Apple MPS      : available (Apple Silicon GPU)")
else:
    print("No GPU detected -> will fall back to CPU")

Python version : 3.13.9
Platform       : macOS-26.4.1-arm64-arm-64bit-Mach-O
PyTorch version: 2.12.0
CUDA available : False
Apple MPS      : available (Apple Silicon GPU)


## 2. Library Imports and Reproducibility

We import the libraries used in this notebook and set random seeds so that runs are reproducible. The `device` variable is the single source of truth for where tensors live during training.

In [2]:
import os
import glob
import random
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Reproducibility
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Choose device once and reuse everywhere
device = (
    "cuda" if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")

Using device: mps


## 3. Downloading the ModelNet10 Dataset

ModelNet10 contains 10 classes of CAD-modelled household objects (bathtub, bed, chair, desk, dresser, monitor, night_stand, sofa, table, toilet). It's the smaller cousin of ModelNet40

In [3]:
import urllib.request
import zipfile

DATA_URL = "http://3dvision.princeton.edu/projects/2014/3DShapeNets/ModelNet10.zip"

# Locate the project root no matter where the notebook is launched from
cwd = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.dirname(cwd) if os.path.basename(cwd) == "notebooks" else cwd

DATA_ROOT = os.path.join(PROJECT_ROOT, "data")
ZIP_PATH  = os.path.join(DATA_ROOT, "ModelNet10.zip")
DATA_DIR  = os.path.join(DATA_ROOT, "ModelNet10")

os.makedirs(DATA_ROOT, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data root   : {DATA_ROOT}")

if not os.path.isdir(DATA_DIR):
    if not os.path.isfile(ZIP_PATH):
        print(f"\nDownloading ModelNet10 from {DATA_URL} ...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
        print("Download complete.")
    print(f"Extracting to {DATA_ROOT} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_ROOT)
    print("Extraction complete.")
else:
    print(f"\nDataset already present at {DATA_DIR} - skipping download.")

print(f"\nDATA_DIR = {DATA_DIR}")
print(f"Exists   : {os.path.isdir(DATA_DIR)}")

Project root: /Users/dosvatsky/3D Object Detection
Data root   : /Users/dosvatsky/3D Object Detection/data

Download complete.
Extracting to /Users/dosvatsky/3D Object Detection/data ...
Extraction complete.

DATA_DIR = /Users/dosvatsky/3D Object Detection/data/ModelNet10
Exists   : True


## 4. Inspect the Dataset Structure



In [4]:
# Each class is a sub-folder (10 of them); ignore any stray README files
class_folders = sorted(
    f for f in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, f))
)
print(f"Number of classes: {len(class_folders)}")
print("Classes:", class_folders)

print("\nFile counts per class:")
print(f"{'class':<14}{'train':>8}{'test':>8}")
print("-" * 30)
total_train = total_test = 0
for c in class_folders:
    n_train = len(glob.glob(os.path.join(DATA_DIR, c, "train", "*.off")))
    n_test  = len(glob.glob(os.path.join(DATA_DIR, c, "test",  "*.off")))
    total_train += n_train
    total_test  += n_test
    print(f"{c:<14}{n_train:>8}{n_test:>8}")
print("-" * 30)
print(f"{'TOTAL':<14}{total_train:>8}{total_test:>8}")

Number of classes: 10
Classes: ['bathtub', 'bed', 'chair', 'desk', 'dresser', 'monitor', 'night_stand', 'sofa', 'table', 'toilet']

File counts per class:
class            train    test
------------------------------
bathtub            106      50
bed                515     100
chair              889     100
desk               200      86
dresser            200      86
monitor            465     100
night_stand        200      86
sofa               680     100
table              392     100
toilet             344     100
------------------------------
TOTAL             3991     908
